In [ ]:
##### PYTHON IMPORTS #####
import os
import time
import tempfile
from tempfile import NamedTemporaryFile
import gc
import logging
import itertools
import warnings
import csv
from configparser import ConfigParser
import psutil
import threading
import sys

from IPython.display import display, Javascript

##### NUMPY & PANDAS IMPORTS #####
import numpy as np
import pandas as pd
from pandas.errors import SettingWithCopyWarning

##### MATPLOTLIB IMPORTS #####
import matplotlib.pyplot as plt

##### RE, STRING & DEMOJI IMPORTS #####
import re
import string
import demoji
import emoji

##### NLTK IMPORTS #####
import nltk
from nltk.stem import SnowballStemmer
from nltk.stem.wordnet import WordNetLemmatizer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize

##### WORDCLOUD IMPORTS #####
from wordcloud import WordCloud

##### TQDM IMPORTS #####
from tqdm import tqdm

##### LANGDETECT IMPORTS #####
from langdetect import detect

##### LANGCODES IMPORTS #####
from langcodes import Language, LanguageTagError

##### DEEP_TRANSLATOR IMPORTS #####
from deep_translator import GoogleTranslator

##### SPACY IMPORTS #####
import spacy

##### PICKLE IMPORTS #####
import pickle

##### TORCH IMPORTS #####
import torch
from torch.utils.data import DataLoader, TensorDataset
from torch.nn import BCEWithLogitsLoss, BCELoss
from torch.optim import Adam
import torch.nn as nn
import torch.nn.functional as F
import torchtext.vocab as vocab

##### TRANSFORMERS IMPORTS #####
from transformers import (AutoTokenizer, AutoModel, pipeline,
                          DistilBertForSequenceClassification, DistilBertTokenizer, 
                          Seq2SeqTrainer, DataCollatorForSeq2Seq, AutoModelForSeq2SeqLM, 
                          AutoModelForSequenceClassification, Seq2SeqTrainingArguments,
                          TFAutoModelForSequenceClassification, AutoConfig)

from sentence_transformers import SentenceTransformer

##### HUGGINGFACE_HUB IMPORTS #####
from huggingface_hub import notebook_login

##### TENSORFLOW & KERAS IMPORTS #####
import tensorflow as tf
from keras.models import Sequential, Model
from keras.layers import (Dense, LSTM, Embedding, Input, concatenate, Flatten, Dropout, 
                          Conv1D, MaxPooling1D, GlobalMaxPooling1D, GlobalAveragePooling1D)

##### SKLEARN IMPORTS #####
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (classification_report, f1_score, roc_auc_score, 
                             accuracy_score, confusion_matrix, ConfusionMatrixDisplay)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.decomposition import PCA, IncrementalPCA
from sklearn.manifold import TSNE


##### IMBALANCE_LEARN IMPORTS #####
from imblearn.under_sampling import RandomUnderSampler

##### GENSIM IMPORTS #####
import gensim  
from gensim.models import Word2Vec
from gensim.models.callbacks import CallbackAny2Vec

##### SCIPY IMPORTS #####
from scipy.special import softmax

##### PLOTLY IMPORTS #####
import plotly.graph_objects as go

##### EVALUATE IMPORTS #####
import evaluate

##### CYTHON IMPORTS #####
import cython

##### SET SEEDS #####
tf.random.set_seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed(42)

##### DEVICE SETUP #####
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

##### NLTK DOWNLOADS #####
nltk.download("punkt")
nltk.download("stopwords")
nltk.download("wordnet")

#### Auxiliary Functions

In [ ]:
# Auxiliary function for confusion matrix plotting

def plot_conf_matrix(y_true, y_pred, labels=["Not Unlisted", "Unlisted"], plot_name="Confusion Matrix"):
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    disp.plot(cmap=plt.cm.Blues)
    plt.title(plot_name)
    plt.show()

In [ ]:
# Function to detect language of the comment

def detect_and_return_language(text):
    try:
        detected_lang_code = detect(text)
        lang = Language.get(detected_lang_code)
        return lang.display_name().lower()
    except ValueError:
        raise ValueError("Language code not found")

In [ ]:
# Function to check if any contraction exists in a text

def contains_contractions(text, contractions_list):
    for contraction in contractions_list:
        if contraction in str(text):
            return True
    return False

contractions_dict = {
    "ain't": "am not",
    "aren't": "are not",
    "can't": "cannot",
    "couldn't": "could not",
    "didn't": "did not",
    "doesn't": "does not",
    "don't": "do not",
    "hadn't": "had not",
    "hasn't": "has not",
    "haven't": "have not",
    "he'd": "he would",
    "he'll": "he will",
    "he's": "he is",
    "I'd": "I would",
    "I'll": "I will",
    "I'm": "I am",
    "I've": "I have",
    "isn't": "is not",
    "it's": "it is",
    "let's": "let us",
    "mustn't": "must not",
    "shan't": "shall not",
    "she'd": "she would",
    "she'll": "she will",
    "she's": "she is",
    "shouldn't": "should not",
    "that's": "that is",
    "there's": "there is",
    "they'd": "they would",
    "they'll": "they will",
    "they're": "they are",
    "they've": "they have",
    "we'd": "we would",
    "we'll": "we will",
    "we're": "we are",
    "we've": "we have",
    "weren't": "were not",
    "what'll": "what will",
    "what're": "what are",
    "what's": "what is",
    "what've": "what have",
    "where's": "where is",
    "who'd": "who would",
    "who'll": "who will",
    "who're": "who are",
    "who's": "who is",
    "who've": "who have",
    "won't": "will not",
    "wouldn't": "would not",
    "you'd": "you would",
    "you'll": "you will",
    "you're": "you are",
    "you've": "you have",
    }
contractions_list = list(contractions_dict.keys())

# Function to expand contractions
def expand_contractions(text):
    # Regular expression pattern to find contractions
    contractions_pattern = re.compile(
        "({})".format("|".join(contractions_dict.keys())),
        flags=re.IGNORECASE | re.DOTALL,
    )

    def expand_match(contraction):
        match = contraction.group(0)
        first_char = match[0]
        expanded_contraction = (
            contractions_dict.get(match)
            if contractions_dict.get(match)
            else contractions_dict.get(match.lower())
        )
        if expanded_contraction is None:
            expanded_contraction = match
        else:
            expanded_contraction = first_char + expanded_contraction[1:]
        return expanded_contraction

    # Replace contractions with their expanded forms
    expanded_text = contractions_pattern.sub(expand_match, text)
    return expanded_text

In [ ]:
# EXTRA METHOD - CONVERTING EMOJIS TO TEXT

def emoji_to_text(text):
    """
    Convert emojis to text
    """

    text = emoji.demojize(text)

    # Replace underscores in emoji descriptions with spaces
    text = re.sub(r'_', ' ', text)
    
    # Insert spaces around the emoji text descriptions (delimited by colons)
    text = re.sub(r'(:\S+?:)', r' \1 ', text)

    # Remove multiple spaces if they occur
    text = re.sub(r'\s+', ' ', text)

    # Remove any spaces at beginning or end
    return text.strip()

In [ ]:
# Function to clean data
# Using regex to remove html tags, money symbols, dates among others

def clean_df(df):

    copy_df = df.copy()
    

    # to better visualise the following regular expressions please visit: https://extendsclass.com/regex-tester.html#python
    html_tag_regex = r"<\W*[a-zA-Z]{1,8}\s*\W*>"

    date_regex = r"\d{2}/\d{2}/(\d{4}|\d{2})|((\d{4}|\d{2})/\d{2}/\d{2})"

    time_regex = (r"\b((\d{1,2}((?:h|H|:|-)\d{2}(min)?)(?:AM|PM)?)|(\d{1,2}\s*(?:AM|PM|h|H)))\b")

    regex_for_monetary_values = r"^(\b((?:€|EUR|\$|USD|GBP|£|JPY|¥)\s*(?:\d+(?:\.\d{1,2})?|\.\d{1,2}))\b|\b((?:\d+(?:\.\d{1,2})?|\.\d{1,2})\s*(?:€|EUR|\$|USD|GBP|£|JPY|¥))\b)$"


    col_index_dict = {}
    # print column name and its index
    for i, col in enumerate(copy_df.columns):
        col_index_dict[col] = i


    for i in tqdm(range(len(copy_df))):
        comment_index = col_index_dict["comments"]
        house_description_index = col_index_dict["description"]
        host_about_index = col_index_dict["host_about"]


        # get the text
        comment = str(copy_df.iloc[i, comment_index])
        house_descrp = str(copy_df.iloc[i, house_description_index])
        host_about = str(copy_df.iloc[i, host_about_index])
        
        # Check for Contractions
        comment_contains_contraction_boolean = col_index_dict["comments_contains_contraction"]
        house_description_contains_contraction_boolean = col_index_dict["description_contains_contraction"]
        host_about_contains_contraction_boolean = col_index_dict["host_about_contains_contraction"]


        # Turning emojis into text - EXTRA PREPROCESSING METHOD
        comment = emoji_to_text(comment)
        house_descrp = emoji_to_text(house_descrp)
        host_about = emoji_to_text(host_about)

        # remove html tags
        comment = re.sub(html_tag_regex, " ", comment)
        house_descrp = re.sub(html_tag_regex, " ", house_descrp)
        host_about = re.sub(html_tag_regex, " ", host_about)

        # remove monetary references
        comment = re.sub(regex_for_monetary_values, "MONETARY_VALUE_STAMP", comment)
        house_descrp = re.sub(regex_for_monetary_values, "MONETARY_VALUE_STAMP", house_descrp)
        host_about = re.sub(regex_for_monetary_values, "MONETARY_VALUE_STAMP", host_about)

        # Change dates to DATE
        comment = re.sub(date_regex, "DATE_STAMP", comment)
        house_descrp = re.sub(date_regex, "DATE_STAMP", house_descrp)
        host_about = re.sub(date_regex, "DATE_STAMP", host_about)

        # remove time references
        comment = re.sub(time_regex, "TIME_STAMP", comment)
        house_descrp = re.sub(time_regex, "TIME_STAMP", house_descrp)
        host_about = re.sub(time_regex, "TIME_STAMP", host_about)

        # remove punctuation
        comment = re.sub(r'[^\w\s]', ' ', comment)
        house_descrp = re.sub(r'[^\w\s]', ' ', house_descrp)
        host_about = re.sub(r'[^\w\s]', ' ', host_about)

        # lower case everything
        comment = comment.lower()
        house_descrp = house_descrp.lower()
        host_about = host_about.lower()

        # Check for Contractions and Expand them
        if comment_contains_contraction_boolean:
            comment = expand_contractions(comment)
        if house_description_contains_contraction_boolean:
            house_descrp = expand_contractions(house_descrp)
        if host_about_contains_contraction_boolean:
            host_about = expand_contractions(host_about)
        
        # Check for 'x000d' in text and remove
        if "x000d" in str(comment):
            comment = comment.replace("x000d", "")
        if "x000d" in str(house_descrp):
            house_descrp = house_descrp.replace("x000d", "")
        if "x000d" in str(host_about):
            host_about = host_about.replace("x000d", "")

        # tokenize
        comment = word_tokenize(comment)
        house_descrp = word_tokenize(house_descrp)
        host_about = word_tokenize(host_about)

        # remove stop words
        stop = set(stopwords.words('english'))
        comment = [word for word in comment if word not in stop]
        house_descrp = [word for word in house_descrp if word not in stop]
        host_about = [word for word in host_about if word not in stop]

        # join back to string
        comment = " ".join(comment)
        house_descrp = " ".join(house_descrp)
        host_about = " ".join(host_about)

        # update dataframe
        copy_df.iloc[i, col_index_dict["comments"]] = comment
        
        copy_df.iloc[i, col_index_dict["description"]] = house_descrp

        copy_df.iloc[i, col_index_dict["host_about"]] = host_about


    # drop contain_contraction columns
    if "comments_contains_contraction" in copy_df.columns:
        copy_df.drop(columns=["comments_contains_contraction"], inplace=True)
    if "description_contains_contraction" in copy_df.columns:
        copy_df.drop(columns=["description_contains_contraction"], inplace=True)
    if "host_about_contains_contraction" in copy_df.columns:
        copy_df.drop(columns=["host_about_contains_contraction"], inplace=True)


    return copy_df

In [ ]:
# Function to merge all comments into same line for each property
# This is done to have all comments in a single cell for each property

def create_df_with_comments_and_target(df):
    """
    Converte os comentários de várias propriedades para uma única string, agrupando-os por listing_id.
    Devolve um dataframe com os comentários agrupados na mesma célula e a variável target.
    Args:
        df (pd.Dataframe): dataframe com os comentários de várias propriedades e a variável target.

    Returns:
        pd.Dataframe: dataframe com os comentários agrupados na mesma célula e a variável target.
    """
    df2 = df.copy()
    df2 = df2.groupby("listing_id").agg({"comments": ",".join, "unlisted": "first"}).reset_index()
    # for every listing_id add host_about and description
    df2["host_about"] = df2["listing_id"].apply(lambda x: df[df["listing_id"] == x]["host_about"].values[0])
    df2["description"] = df2["listing_id"].apply(lambda x: df[df["listing_id"] == x]["description"].values[0])
    return df2



# Same function but for test data

def create_df_with_comments(df):
    """O mesmo que em cima, mas para o test que n tem target
    PARA O DATASET DE PREDICTIONS!

    Args:
        df (pd.Dataframe): dataframe com os comentários de várias propriedades

    Returns:
        pd.Dataframe: dataframe com os comentários agrupados na mesma célula
    """
    df2 = df.copy()
    df2 = df2.groupby("listing_id").agg({"comments": ",".join, }).reset_index()
    # for every listing_id add host_about and description
    df2["host_about"] = df2["listing_id"].apply(lambda x: df[df["listing_id"] == x]["host_about"].values[0])
    df2["description"] = df2["listing_id"].apply(lambda x: df[df["listing_id"] == x]["description"].values[0])
    return df2

In [ ]:
os.chdir(os.path.join("Project Corpora"))

In [ ]:
#raw datasets
raw_train = pd.read_csv("train.csv")
raw_train_reviews = pd.read_csv("train_reviews.csv")

# Data Exploration - before preprocessing

In [ ]:
raw_train.head(10)  

In [ ]:
raw_train_reviews.head(10)

In [ ]:
raw_train_reviews.comments[0]

In [ ]:
# Rename 'index' column to 'listing_id'
raw_train_reviews.rename(columns={"index": "listing_id"}, inplace=True)
raw_train.rename(columns={"index": "listing_id"}, inplace=True)

In [ ]:
raw_train.shape

In [ ]:
raw_train_reviews.shape

In [ ]:
raw_train.isna().sum()

In [ ]:
raw_train_reviews.isna().sum()

We need to drop NAs before converting column types otherwise they will stop being considered as NA and dropping will not work.<br>
We drop them bcause they are just 2. If they were many more we wouldn't drop them.

In [ ]:
raw_train_reviews.dropna(subset=["comments"], inplace=True)

In [ ]:
raw_train_reviews.isna().sum()

In [ ]:
listing_correct_types = {"description": str, "host_about": str}

reviews_correct_types = {"comments": str}

raw_train = raw_train.astype(listing_correct_types)
raw_train_reviews = raw_train_reviews.astype(reviews_correct_types)

### Unlisted

In [ ]:
raw_train["unlisted"].unique()

In [ ]:
plt.bar(
    raw_train["unlisted"].value_counts().index,
    raw_train["unlisted"].value_counts().values,
)
plt.grid(False)
plt.title("Bar Chart of Listing Status")
plt.xlabel("Listing Status")
plt.ylabel("Frequency")
plt.xticks(np.arange(0,2))
plt.gca().spines["top"].set_visible(False)
plt.gca().spines["right"].set_visible(False)
plt.show()

### Description

In [ ]:
# Get size of description and order by biggest to smallest
raw_train["description"].apply(len).value_counts()

In [ ]:
# Get word count of description and order by biggest to smallest
raw_train["description_word_count"] = raw_train["description"].apply(lambda x: len(str(x).split(" ")))

raw_train["description_word_count"].value_counts()

In [ ]:
raw_train[raw_train["description_word_count"] == 3]["description"]

In [ ]:
raw_train["description_word_count"].describe()

In [ ]:
raw_train.boxplot(column=["description_word_count"])
plt.title("Boxplot of Scores")
plt.gca().spines["top"].set_visible(False)
plt.gca().spines["right"].set_visible(False)
plt.gca().spines["bottom"].set_visible(False)
plt.gca().spines["left"].set_visible(False)
plt.show()

In [ ]:
raw_train["description_word_count"].hist()
plt.grid(False)
plt.title("Distribution of Description Word Count")
plt.xlabel("Score")
plt.ylabel("Frequency")
plt.gca().spines["top"].set_visible(False)
plt.gca().spines["right"].set_visible(False)
plt.show()

In [ ]:
all_words = " ".join(raw_train["description"].str.lower()).split()

freq = pd.Series(all_words).value_counts()

In [ ]:
x_labels = freq.index[0:10]
values = freq[:10]
plt.bar(x_labels, values)
plt.xticks(x_labels)
plt.ylabel("Frequency")
plt.title("Most Frequent Words")
plt.gca().spines["top"].set_visible(False)
plt.gca().spines["right"].set_visible(False)
plt.show()

### Comments

In [ ]:
comment_counts = raw_train_reviews.groupby("listing_id")["comments"].count()

# Sort the counts in descending order
sorted_comment_counts = comment_counts.sort_values(ascending=False)

# Display the top 10 indices with the most comments
top_10_indices = sorted_comment_counts.head(10)
top_10_indices

In [ ]:
# get character count of comments
raw_train_reviews["comment_character_count"] = raw_train_reviews["comments"].apply(len)
raw_train_reviews["comment_character_count"].value_counts()

In [ ]:
# from 3 characters onwards, the comments seem to be valid
raw_train_reviews[raw_train_reviews["comment_character_count"] == 3]["comments"].value_counts()

In [ ]:
raw_train_reviews.boxplot(column=["comment_character_count"])
plt.title("Boxplot of Scores")
plt.gca().spines["top"].set_visible(False)
plt.gca().spines["right"].set_visible(False)
plt.gca().spines["bottom"].set_visible(False)
plt.gca().spines["left"].set_visible(False)
plt.show()

In [ ]:
freq = pd.Series(raw_train_reviews[raw_train_reviews["comment_character_count"] == 1]["comments"].value_counts())

In [ ]:
plt.bar(freq.index[:10], freq[:10])
plt.grid(False)
plt.title("Most common characters in one word comments")
plt.xlabel("Characters")
plt.ylabel("Frequency")
plt.gca().spines["top"].set_visible(False)
plt.gca().spines["right"].set_visible(False)
plt.show();

In [ ]:
# get word count of comments
raw_train_reviews["comment_word_count"] = raw_train_reviews["comments"].apply(lambda x: len(str(x).split(" ")))
raw_train_reviews["comment_word_count"].value_counts()

In [ ]:
# Even 1-worded comments seem valid (apart from non-alphabetical ones, obviously)
raw_train_reviews[raw_train_reviews["comment_word_count"] == 1]["comments"][:10]

In [ ]:
raw_train_reviews.boxplot(column=["comment_word_count"])
plt.title("Boxplot of Scores")
plt.gca().spines["top"].set_visible(False)
plt.gca().spines["right"].set_visible(False)
plt.gca().spines["bottom"].set_visible(False)
plt.gca().spines["left"].set_visible(False)
plt.show()

In [ ]:
all_comment_words = " ".join(raw_train_reviews["comments"].str.lower()).split()

freq = pd.Series(all_comment_words).value_counts().sort_values(ascending=False)

In [ ]:
x_labels = freq.index[0:10]
values = freq[:10]
plt.bar(x_labels, values)
plt.xticks(x_labels)
plt.ylabel("Frequency")
plt.title("Most Frequent Words")
plt.gca().spines["top"].set_visible(False)
plt.gca().spines["right"].set_visible(False)
plt.show()

In [ ]:
# check for duplicates by 'listing_id' and 'comments', and show rows
duplicated_rows = raw_train_reviews[raw_train_reviews.duplicated(["listing_id", "comments"])]

duplicated_rows[["listing_id", "comments", "comment_word_count"]].sort_values("comment_word_count", ascending=False)[:10]

In [ ]:
# Joining all the comments into a single string
all_reviews = " ".join(raw_train_reviews["comments"])

# Create WordCloud object
wordcloud = WordCloud(width=800, height=400, background_color="white").generate(all_reviews)

# Plotting the word cloud
plt.figure(figsize=(10, 5))
plt.imshow(wordcloud)
plt.axis("off")
plt.show();

### Host_about

In [ ]:
# get size of host_about
raw_train["host_about_character_count"] = raw_train["host_about"].apply(len)
raw_train["host_about_character_count"].value_counts()

In [ ]:
raw_train[raw_train["host_about_character_count"] == 6]["host_about"]

In [ ]:
raw_train.boxplot(column=["host_about_character_count"])
plt.title("Boxplot of Scores")
plt.gca().spines["top"].set_visible(False)
plt.gca().spines["right"].set_visible(False)
plt.gca().spines["bottom"].set_visible(False)
plt.gca().spines["left"].set_visible(False)
plt.show()

We decided not to show a bar plot of the `host_about_character_count` as there are many outliers (as seen above) and it would make the plot unreadable.

In [ ]:
raw_train["host_about_word_count"] = raw_train["host_about"].apply(lambda x: len(str(x).split(" ")))

raw_train["host_about_word_count"].value_counts()

In [ ]:
# It's ok to say even from 1-worded host_about, it's still valid (apart from non-alphabetical ones, obviously)
raw_train[raw_train["host_about_word_count"] == 1]["host_about"]

In [ ]:
raw_train.boxplot(column=["host_about_word_count"])
plt.title("Boxplot of 'host_about' word count")
plt.gca().spines["top"].set_visible(False)
plt.gca().spines["right"].set_visible(False)
plt.gca().spines["bottom"].set_visible(False)
plt.gca().spines["left"].set_visible(False)
plt.show()

In [ ]:
raw_train["host_about_word_count"].hist()
plt.grid(False)
plt.title("Distribution of Description Word Count")
plt.xlabel("Number of Words")
plt.ylabel("Frequency")
plt.gca().spines["top"].set_visible(False)
plt.gca().spines["right"].set_visible(False)
plt.show()

In [ ]:
all_words = " ".join(raw_train["host_about"].str.lower()).split()

freq = pd.Series(all_words).value_counts()

In [ ]:
x_labels = freq.index[0:10]
values = freq[:10]
plt.bar(x_labels, values)
plt.xticks(x_labels)
plt.ylabel("Frequency")
plt.title("Most Frequent Words")
plt.gca().spines["top"].set_visible(False)
plt.gca().spines["right"].set_visible(False)
plt.show()

##### We're going to check how many comments from each language we have

In [ ]:
def detect_language(text):
    try:
        return detect(text)
    except:
        return 'Error'  # In case text is too short or any other issue

In [ ]:
# Apply the language detection to the 'comment' column
tqdm.pandas()

raw_train_reviews["language"] = raw_train_reviews["comments"].progress_apply(detect_language)

raw_train_reviews.to_csv("train_reviews_with_langs.csv", index = False)

In [ ]:
# Count the number of comments in each language
raw_train_reviews = pd.read_csv("train_reviews_with_langs.csv")

language_counts = raw_train_reviews["language"].value_counts()

print(language_counts)

# Data Preprocessing

#### Note - **EXTRA METHODS**: Translation to English, Emoji to Text Conversion

First, we translated all non-english `description` and `host about` to english on the `train.xlsx` file. We also translated to english the `comments` on the `train_reviews.xlsx` file. We applied the exact same procedure for the `test.xlsx` and `test_reviews.xlsx`, since they follow the same structure.

-- TO-DO: Use only `raw_train_reviews` and subset comments that are not in english for translation process

#### Translation 


In [ ]:
warnings.simplefilter(action="ignore", category=SettingWithCopyWarning)

In [ ]:
train= pd.read_csv("train.csv")
train_reviews = pd.read_csv("train_reviews.csv")
test = pd.read_csv("test.csv")
test_reviews = pd.read_csv("test_reviews.csv")

Basic curation of rows with empty values

In [ ]:
train.isna().sum()

In [ ]:
train_reviews.isna().sum()

In [ ]:
train_reviews.dropna(subset = "comments", inplace = True) # Just 2 comments were dropped, not worrisome
train_reviews.reset_index(inplace = True, drop = True)

In [ ]:
train_reviews.isna().sum()

In [ ]:
test.isna().sum()

In [ ]:
test_reviews.isna().sum()

In [ ]:
train.head(10)

In [ ]:
# Apply the language detection to the 'comment' column
tqdm.pandas()

train["language_description"] = train["description"].progress_apply(detect_language)
train["language_host_about"] = train["host_about"].progress_apply(detect_language)

In [ ]:
train.to_csv("train_with_langs.csv", index = False)

`train` Translation

In [ ]:
error = 0

train["description_en"] = np.nan
train["host_about_en"] = np.nan

columns = ["index", "description", "description_en", "host_about","host_about_en", "unlisted"]

train = train[columns]

In [ ]:
train.head()

In [ ]:
for i in tqdm(range(len(train))):
    try:
        # Attempt to translate the comment
        train["description_en"][i] = GoogleTranslator(source='auto', target='en').translate(text=train["description"][i])
        train["host_about_en"][i] = GoogleTranslator(source='auto', target='en').translate(text=train["host_about"][i])
    except Exception as e:
        # Log or handle translation errors
        error += 1
        train['description_en'][i]= train["description"][i]
        train["host_about_en"][i] = train["host_about"][i]
        
        print(f"Translation failed at index {i}: {e}")

print(f"Translation ended with {error} errors.")

In [ ]:
train.to_csv("train_en.csv", index = False)

In [ ]:
train.head()

`train_reviews` Translation

In [ ]:
train_reviews.head()

In [ ]:
error = 0

train_reviews["comments_en"] = np.nan

In [ ]:
for i in tqdm(range(len(train_reviews))):
    try:
        # Attempt to translate the comment
        train_reviews["comments_en"][i] = GoogleTranslator(source='auto', target='en').translate(text=train_reviews["comments"][i])
    except Exception as e:
        # Log or handle translation errors
        error += 1
        train_reviews['comments_en'][i]= 'Error'
        
        print(f"Translation failed at index {i}: {e}")

print(f"Translation ended with {error} errors.")

In [ ]:
train_reviews.to_csv("train_reviews_en.csv", index = False)

In [ ]:
train_reviews.head()

`test` Translation

In [ ]:
test.head()

In [ ]:
error = 0

test["description_en"] = np.nan
test["host_about_en"] = np.nan

columns = ["index", "description", "description_en", "host_about","host_about_en"]

test = test[columns]

In [ ]:
for i in tqdm(range(len(test))):
    try:
        # Attempt to translate the comment
        test["description_en"][i] = GoogleTranslator(source='auto', target='en').translate(text=test["description"][i])
        test["host_about_en"][i] = GoogleTranslator(source='auto', target='en').translate(text=test["host_about"][i])
    except Exception as e:
        # Log or handle translation errors
        error += 1
        test['description_en'][i]= test["description"][i]
        test["host_about_en"][i] = test["host_about"][i]
        
        print(f"Translation failed at index {i}: {e}")

print(f"Translation ended with {error} errors.")

In [ ]:
test.to_csv("test_en.csv", index = False)

In [ ]:
test.head()

`test_reviews` Translation

In [ ]:
test_reviews.head(10)

In [ ]:
error = 0

test_reviews["comments_en"] = np.nan

In [ ]:
for i in tqdm(range(len(test_reviews))):
    try:
        # Attempt to translate the comment
        test_reviews["comments_en"][i] = GoogleTranslator(source='auto', target='en').translate(text=test_reviews["comments"][i])
    except Exception as e:
        # Log or handle translation errors
        error += 1
        test_reviews['comments_en'][i]= 'Error'
        
        print(f"Translation failed at index {i}: {e}")

print(f"Translation ended with {error} errors.")

In [ ]:
test_reviews.to_csv("test_reviews_en.csv", index = False)

In [ ]:
test_reviews.head()

In [ ]:
train = pd.read_csv("train_en.csv")
train_reviews = pd.read_csv("train_reviews_en.csv")
test = pd.read_csv("test_en.csv")
test_reviews = pd.read_csv("test_reviews_en.csv")

During the translation, there were some translation errors. TThey were all associated with test length (Google Translator API only supports up to 5000 characters of text length).
<br>
However, the datasets are so big and the errors are so few in comparison, that we decided that for the `train_reviews` (length 361279) we would drop the rows where the translation output an error. The number of rows dropped is just 21 (0.0058% of the dataset).

In [ ]:
train.head()

In [ ]:
train_reviews.head()

In [ ]:
# Dropping the non-translated comments columns
train_reviews.drop(columns = {"comments"}, inplace = True)
test_reviews.drop(columns = {"comments"}, inplace = True)

In [ ]:
# Changing index to listing_id for easier identification of what it means
train_reviews = train_reviews.rename(columns={"index" : "listing_id"})
test_reviews = test_reviews.rename(columns={"index" : "listing_id"})
train = train.rename(columns={"index" : "listing_id"})
test = test.rename(columns={"index" : "listing_id"})

In [ ]:
# Changing `comments_en` to `comments`
train_reviews = train_reviews.rename(columns={"comments_en" : "comments"})
test_reviews = test_reviews.rename(columns={"comments_en" : "comments"})

In [ ]:
train.head()

In [ ]:
test.head()

In [ ]:
df_train = pd.merge(train_reviews, train, on = "listing_id", how = "outer")

In [ ]:
df_train.duplicated().sum()

In [ ]:
df_train.drop_duplicates(inplace = True)

In [ ]:
df_train.duplicated().sum()

In [ ]:
train.shape

In [ ]:
train_reviews.shape

In [ ]:
df_train.shape

In [ ]:
# Reordering columns
columns = ["listing_id", "host_about_en", "description_en", "comments", "unlisted"]

df_train = df_train[columns]

df_train.rename(columns={"host_about_en" : "host_about", "description_en" : "description"}, inplace = True)

In [ ]:
df_train.head(10)

In [ ]:
df_train["comments_contains_contraction"] = df_train["comments"].apply(
    lambda x: contains_contractions(x, contractions_list)
)
df_train["description_contains_contraction"] = df_train["description"].apply(
    lambda x: contains_contractions(x, contractions_list)
)
df_train["host_about_contains_contraction"] = df_train["host_about"].apply(
    lambda x: contains_contractions(x, contractions_list)
)

In [ ]:
df_train.isna().sum()

We find that 4513 `host_about` are empty and 2732 `comments` are empty. Since we already merged with `train`, we won't drop these, but instead change them fron `NaN` to a default string, for ease of manipulation. This decision is because there might be properties with no comments, and the properties with empty `host_about` might still be useful.

In [ ]:
values = {"comments": "Empty string", "host_about": "Empty string"}
df_train = df_train.fillna(value=values)

In [ ]:
df_train.isna().sum()

In [ ]:
df_train.reset_index(inplace = True, drop = True)

#### Applying same preprocessing for `test`and `test_reviews`

In [ ]:
df_test = pd.merge(test_reviews, test, on = "listing_id", how = "outer")

In [ ]:
df_test.duplicated().sum()

In [ ]:
df_test.drop_duplicates(inplace = True)

In [ ]:
df_test.duplicated().sum()

In [ ]:
df_test["comments_contains_contraction"] = df_test["comments"].apply(
    lambda x: contains_contractions(x, contractions_list)
)

df_test["description_contains_contraction"] = df_test["description"].apply(
    lambda x: contains_contractions(x, contractions_list)
) 

df_test["host_about_contains_contraction"] = df_test["host_about"].apply(
    lambda x: contains_contractions(x, contractions_list)
)

In [ ]:
df_test.isna().sum()

We don't drop anything as this is our test set.

In [ ]:
values = {"comments": "Empty string", "host_about_en": "Empty string"}
df_test = df_test.fillna(value=values)

In [ ]:
df_test.isna().sum()

In [ ]:
test.shape

In [ ]:
test_reviews.shape

In [ ]:
df_test.shape

In [ ]:
# Reordering columns
columns = ["listing_id", "host_about_en", "description_en", "comments", "comments_contains_contraction", "description_contains_contraction", "host_about_contains_contraction"]

df_test = df_test[columns]

df_test.rename(columns={"host_about_en" : "host_about", "description_en" : "description"}, inplace = True)

In [ ]:
df_test.reset_index(inplace = True, drop = True)

In [ ]:
df_test.shape

In [ ]:
df_test.head(10)

In [ ]:
df_train = clean_df(df_train)

In [ ]:
df_train.head(10)

In [ ]:
df_train.to_csv("train_clean.csv", index = False)

In [ ]:
df_train = pd.read_csv("train_clean.csv")

In [ ]:
df_train.isna().sum()

In [ ]:
values = {"comments": "Empty string", "host_about": "Empty string"}
df_train.fillna(value = values, inplace=True)

In [ ]:
df_train.isna().sum()

#### Merging all comments for each listing_id to the same line

In [ ]:
df_train = create_df_with_comments_and_target(df_train)

In [ ]:
df_train.head(10)

In [ ]:
columns = ["listing_id", "host_about", "description", "comments", "unlisted"]
df_train = df_train[columns]

In [ ]:
df_train.isna().sum()

In [ ]:
df_train.unlisted.value_counts(normalize=True)

In [ ]:
df_train.to_csv("train_clean_merged.csv", index = False)

In [ ]:
df_test = clean_df(df_test)

In [ ]:
df_test.isna().sum()

In [ ]:
df_test.head(10)

In [ ]:
df_test.to_csv("test_clean.csv", index = False)

In [ ]:
df_test = create_df_with_comments(df_test)

In [ ]:
df_test.head(10)

In [ ]:
columns = ["listing_id", "host_about", "description", "comments"]
df_test = df_test[columns]

In [ ]:
df_test.to_csv("test_clean_merged.csv", index = False)

## Data Preprocessing - Lemmatization

In [ ]:
lemma = WordNetLemmatizer()

In [ ]:
def lemmatization(text_list):
    updated=[]

    for text in tqdm(text_list):
        text = " ".join(lemma.lemmatize(word) for word in text.split())

        updated.append(text)

    return updated

In [ ]:
df_train = pd.read_csv("train_clean_merged.csv")
df_test = pd.read_csv("test_clean_merged.csv")

In [ ]:
df_train.head()

In [ ]:
df_train.isna().sum()

In [ ]:
df_test.head()

In [ ]:
df_test.isna().sum()

In [ ]:
def update_df(dataframe, list_updated):
    dataframe.update(pd.DataFrame({"comments": list_updated}))

In [ ]:
updates = lemmatization(df_train["comments"])

update_df(df_train, updates)

In [ ]:
df_train.isna().sum()

In [ ]:
df_train.to_csv("train_clean_merged_lm.csv", index = False)

In [ ]:
updates = lemmatization(df_test["comments"])

update_df(df_test, updates)

In [ ]:
df_test.isna().sum()

In [ ]:
df_test.to_csv("test_clean_merged_lm.csv", index = False)

Reading Lemmatized .csv files

In [ ]:
df_train_lm = pd.read_csv("train_clean_merged_lm.csv")
df_test_lm = pd.read_csv("train_clean_merged_lm.csv")

In [ ]:
df_train_lm.head()

In [ ]:
df_train_lm.isna().sum()

In [ ]:
df_test_lm.head()

In [ ]:
df_test_lm.isna().sum()

## Data Preprocessing - Stemming

In [ ]:
stemmer = SnowballStemmer('english')

In [ ]:
def stemming(text_list):
    updated=[]

    for text in tqdm(text_list):
        text = " ".join(stemmer.stem(word) for word in text.split())

        updated.append(text)

    return updated

In [ ]:
# Need to reload the dataframes from .csv because the previous ones are lemmatized
df_train = pd.read_csv("train_clean_merged.csv")
df_test = pd.read_csv("test_clean_merged.csv")

In [ ]:
df_train.isna().sum()

In [ ]:
updates = stemming(df_train["comments"])

update_df(df_train, updates)

In [ ]:
df_train.isna().sum()

In [ ]:
df_train.to_csv("train_clean_merged_stm.csv", index = False)

In [ ]:
df_train.head()

In [ ]:
df_test.isna().sum()

In [ ]:
updates = stemming(df_test["comments"])

update_df(df_test, updates)

In [ ]:
df_test.isna().sum()

In [ ]:
df_test.to_csv("test_clean_merged_stm.csv", index = False)

In [ ]:
df_test.head()

Reading Stemmed .csv files

In [ ]:
df_train_stm = pd.read_csv("train_clean_merged_stm.csv")
df_test_stm = pd.read_csv("test_clean_merged_stm.csv")

In [ ]:
df_train_stm.head()

In [ ]:
df_train_stm.isna().sum()

In [ ]:
df_test_stm.head()

In [ ]:
df_test_stm.isna().sum()

# Data Exploration - After Preprocessing

>For the data exploration after preprocessing, we will use the clean but unmerged dataset (i.e., the one where the comments haven't all been merged into one row for every property)

In [ ]:
df_clean = pd.read_csv("train_clean.csv")

In [ ]:
df_clean.fillna("", inplace = True) # for exploration purposes, we don't mind filling the NaN with an empty string.

### Unlisted

In [ ]:
df_clean["unlisted"].unique()

In [ ]:
df_clean.shape

In [ ]:
plt.bar(
    df_clean["unlisted"].value_counts().index,
    df_clean["unlisted"].value_counts().values,
)
plt.grid(False)
plt.title("Bar Chart of Listing Status")
plt.xlabel("Listing Status")
plt.ylabel("Frequency")
plt.xticks(np.arange(0,2))
plt.gca().spines["top"].set_visible(False)
plt.gca().spines["right"].set_visible(False)
plt.show()

### Description

In [ ]:
# Get size of description and order by biggest to smallest
df_clean["description"].apply(len).value_counts()

In [ ]:
# Get Description word count and order by biggest to smallest
df_clean["description_word_count"] = df_clean["description"].apply(
    lambda x: len(str(x).split(" "))
)
df_clean["description_word_count"].value_counts()

In [ ]:
df_clean[df_clean["description_word_count"] == 3]["description"]

In [ ]:
df_clean["description_word_count"].describe()

In [ ]:
df_clean.boxplot(column=["description_word_count"])
plt.title("Boxplot of Word Count in Description")
plt.gca().spines["top"].set_visible(False)
plt.gca().spines["right"].set_visible(False)
plt.gca().spines["bottom"].set_visible(False)
plt.gca().spines["left"].set_visible(False)
plt.show()

In [ ]:
df_clean["description_word_count"].hist()
plt.grid(False)
plt.title("Distribution of Description Word Count")
plt.xlabel("Number of Words")
plt.ylabel("Frequency")
plt.gca().spines["top"].set_visible(False)
plt.gca().spines["right"].set_visible(False)
plt.show()

> These plots give us visually the information we already had on the previous cell with the `.describe()` method.

##### Most Frequent Words in Description

In [ ]:
all_words = " ".join(df_clean["description"].str.lower()).split()

freq = pd.Series(all_words).value_counts()

In [ ]:
x_labels = freq.index[0:10]
values = freq[:10]
plt.bar(x_labels, values)
plt.xticks(x_labels, rotation = -45)
plt.ylabel("Frequency")
plt.title("Most Frequent Words")
plt.gca().spines["top"].set_visible(False)
plt.gca().spines["right"].set_visible(False)
plt.show()

### Comments

To explore comments, we will first see how many empty comment rows we have.

In [ ]:
df_clean["comments"].isna().sum()

In [ ]:
comment_counts = df_clean.groupby("listing_id")["comments"].count()

# Sort the counts in descending order
sorted_comment_counts = comment_counts.sort_values(ascending=False)

# Display the top 10 indices with the most comments
top_10_indices = sorted_comment_counts.head(10)
top_10_indices

In [ ]:
# get character count of comments
df_clean["comment_character_count"] = df_train_lm["comments"].apply(len)

df_clean["comment_character_count"].value_counts()

In [ ]:
df_clean[df_clean["comment_character_count"] == 3]["comments"].value_counts()

In [ ]:
df_clean.boxplot(column=["comment_character_count"])
plt.title("Distribution of comment character count")
plt.gca().spines["top"].set_visible(False)
plt.gca().spines["right"].set_visible(False)
plt.gca().spines["bottom"].set_visible(False)
plt.gca().spines["left"].set_visible(False)
plt.show()

In [ ]:
# get word count of comments
df_clean["comment_word_count"] = df_clean["comments"].apply(lambda x: len(str(x).split(" ")))

df_clean["comment_word_count"].value_counts()

In [ ]:
# Even 1-worded comments seem valid (apart from non-alphabetical ones, obviously)
df_clean[df_clean["comment_word_count"] == 1]["comments"][:]

The preprocessing significantly reduced the number of 1 word comments existing. We only find 1. This is because our dataset is now merged.

In [ ]:
df_clean.boxplot(column=["comment_word_count"])
plt.title("Boxplot of Scores")
plt.gca().spines["top"].set_visible(False)
plt.gca().spines["right"].set_visible(False)
plt.gca().spines["bottom"].set_visible(False)
plt.gca().spines["left"].set_visible(False)
plt.show()

In [ ]:
all_comment_words = " ".join(df_clean["comments"].str.lower()).split()

freq = pd.Series(all_comment_words).value_counts().sort_values(ascending=False)

In [ ]:
x_labels = freq.index[0:10]
values = freq[:10]
plt.bar(x_labels, values)
plt.xticks(x_labels, rotation = -45)
plt.ylabel("Frequency")
plt.title("Most Frequent Words in comments")
plt.gca().spines["top"].set_visible(False)
plt.gca().spines["right"].set_visible(False)
plt.show()

In [ ]:
# check for duplicated columns by 'listing_id' and 'comments', and show rows
duplicated_rows = df_clean[df_clean.duplicated(["listing_id", "comments"])]

duplicated_rows[["listing_id", "comments", "comment_word_count"]].sort_values("comment_word_count", ascending=False)[:10]

With preprocessing, we got rid of duplicates.

In [ ]:
# Joining all the comments into a single string
all_reviews = " ".join(df_clean["comments"])

# Create WordCloud object
wordcloud = WordCloud(width=800, height=400, background_color="white").generate(all_reviews)

# Plotting the word cloud
plt.figure(figsize=(10, 5))
plt.imshow(wordcloud)
plt.axis("off")
plt.show();

### Host_about

In [ ]:
# Get character size of host_about
df_clean["host_about_character_count"] = df_clean["host_about"].apply(len)

df_clean["host_about_character_count"].value_counts()

In [ ]:
df_clean[df_clean["host_about_character_count"] == 6]["host_about"]

In [ ]:
df_clean.boxplot(column=["host_about_character_count"])
plt.title("Boxplot of Scores")
plt.gca().spines["top"].set_visible(False)
plt.gca().spines["right"].set_visible(False)
plt.gca().spines["bottom"].set_visible(False)
plt.gca().spines["left"].set_visible(False)
plt.show()

We decided not to show a bar plot of the `host_about_character_count` as there are many outliers (as seen above) and it would make the plot unreadable.

In [ ]:
df_clean["host_about_word_count"] = df_clean["host_about"].apply(lambda x: len(str(x).split(" ")))

df_clean["host_about_word_count"].value_counts()

In [ ]:
# I think it's ok to say even from 1-worded host_about, it's still valid (apart from non-alphabetical ones, obviously)
df_clean[df_clean["host_about_word_count"] == 1]["host_about"]

In [ ]:
df_clean.boxplot(column=["host_about_word_count"])
plt.title("Boxplot of Scores")
plt.gca().spines["top"].set_visible(False)
plt.gca().spines["right"].set_visible(False)
plt.gca().spines["bottom"].set_visible(False)
plt.gca().spines["left"].set_visible(False)
plt.show()

In [ ]:
df_clean["host_about_word_count"].hist()
plt.grid(False)
plt.title("Distribution of Description Word Count")
plt.xlabel("Score")
plt.ylabel("Frequency")
plt.gca().spines["top"].set_visible(False)
plt.gca().spines["right"].set_visible(False)
plt.show()

In [ ]:
all_words = " ".join(df_clean["host_about"].str.lower()).split()

freq = pd.Series(all_words).value_counts()

In [ ]:
x_labels = freq.index[0:10]
values = freq[:10]
plt.bar(x_labels, values)
plt.xticks(x_labels, rotation = -45)
plt.ylabel("Frequency")
plt.title("Most Frequent Words in 'Host About'")
plt.gca().spines["top"].set_visible(False)
plt.gca().spines["right"].set_visible(False)
plt.show()

Checking for duplicated `host_about` descriptions are unnecessary as hosts may have more than 1 listing.

# Feature Engineering

### Train-Test Split for **Lemmatized** data

##### **Note**: Although the train-test split is usually the first thing to be done, in our case we did not apply any methods to the data that would lead to data leakage. Therefore, it is fine to apply the train_test_split after the rest of the data preprocessing

In [ ]:
df_train_lm = pd.read_csv("train_clean_merged_lm.csv")

In [ ]:
X_data = df_train_lm.drop(columns=["unlisted"])
y_data = df_train_lm["unlisted"]
df_train, df_test, y_train, y_test = train_test_split(X_data, y_data, test_size=0.3, stratify=y_data, random_state=42)

In [ ]:
df_train["unlisted"] = y_train
df_test["unlisted"] = y_test

In [ ]:
# check proportion of unlisted after spliting
df_train["unlisted"].value_counts(normalize=True)

Proportion holds after splitting.

In [ ]:
df_train.to_csv("df_train_split_lm.csv", index = False)
df_test.to_csv("df_test_split_lm.csv", index = False)

### Train-Test Split for **Stemmed** data

In [ ]:
df_train_stm = pd.read_csv("train_clean_merged_stm.csv")

In [ ]:
X_data = df_train_stm.drop(columns=["unlisted"])
y_data = df_train_stm["unlisted"]
df_train, df_test, y_train, y_test = train_test_split(X_data, y_data, test_size=0.3, stratify=y_data, random_state=42)

In [ ]:
df_train["unlisted"] = y_train
df_test["unlisted"] = y_test

In [ ]:
# check proportion of unlisted after spliting
df_train["unlisted"].value_counts(normalize=True)

Proportion holds after splitting.

In [ ]:
df_train.to_csv("df_train_split_stm.csv", index = False)
df_test.to_csv("df_test_split_stm.csv", index = False)

### Word2Vec Implementation (with lemmatized data)

#### Corpus definition

In [ ]:
df_train = pd.read_csv("train_clean_merged_lm.csv")

In [ ]:
def sentence_to_wordlist(raw):
    words = word_tokenize(raw)  # Tokenize words in each sentence
    return words

# Tokenize each comment into sentences, then words, and flatten into a single list
corpus = [sentence_to_wordlist(sentence) for comment in df_train['comments'] for sentence in sent_tokenize(comment)]

In [ ]:
loss_list = []
loss_list.append(0)

logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s', level=logging.INFO)

class Callback(CallbackAny2Vec):
    def __init__(self):
        self.epoch = 0

        def on_epoch_end(self, model):
            loss = model.get_latest_training_loss()
            now_loss = loss - loss_list[-1]
            loss_list.append(loss)
            print(f"Loss after epoch {self.epoch}: {now_loss}")
            self.epoch = self.epoch + 1

In [ ]:
w2v = Word2Vec(corpus, min_count = 10, vector_size = 400, sg = 1, compute_loss = True, workers = 4, seed = 42, epochs = 500,  callbacks=[Callback()]) 

In [ ]:
# Saving the model
with tempfile.NamedTemporaryFile(prefix='gensim-model-', delete=False) as tmp:
    temporary_filepath = tmp.name
    w2v.save(temporary_filepath)

In [ ]:
# # Load model
# w2v = gensim.models.Word2Vec.load('C:\\Users\\andre\\OneDrive\\Documentos\\Nova IMS\\1º ano\\2º Semestre\\Text Mining\\Project\\Project Corpora\\w2v_model_final')

In [ ]:
# Auxiliary function

def reduce_dimensions(model):
    num_dimensions = 2  # final num dimensions (2D, 3D, etc)

    # extract the words & their vectors, as numpy arrays
    vectors = np.asarray(model.wv.vectors)
    labels = np.asarray(model.wv.index_to_key)  # fixed-width numpy strings

   # # Convert numpy strings to standard Python strings (maybe not needed)
    #labels = [str(label) for label in labels]

    # reduce using t-SNE
    tsne = TSNE(n_components=num_dimensions, random_state=42)
    vectors = tsne.fit_transform(vectors)

    x_vals = [v[0] for v in vectors]
    y_vals = [v[1] for v in vectors]
    return x_vals, y_vals, labels

In [ ]:
# Auxiliary function

def plot_with_plotly(x_vals, y_vals, labels, plot_in_notebook=True):
    from plotly.offline import init_notebook_mode, iplot, plot
    import plotly.graph_objs as go

    # Define the trace
    trace = go.Scatter(x=x_vals, y=y_vals, mode='markers', text=labels, hoverinfo='text')
    data = [trace]
    
    # Define the layout
    layout = go.Layout(
        title='Hover over points to see labels',
        xaxis=dict(title='X Axis Label'),
        yaxis=dict(title='Y Axis Label'),
        hovermode='closest'  # Ensures the hover is for the closest point
    )

    # Create a figure with data and layout
    fig = go.Figure(data=data, layout=layout)

    # Plot in notebook or create HTML file
    if plot_in_notebook:
        init_notebook_mode(connected=True)
        iplot(fig)
    else:
        plot(fig, filename='plot.html')  # Outputs an HTML file with the plot

In [ ]:
x_vals, y_vals, labels = reduce_dimensions(w2v)

In [ ]:
plot_with_plotly(x_vals, y_vals, labels, plot_in_notebook=True)

### Implementing Transformer: Roberta - `j-hartmann/emotion-english-roberta-large`: https://huggingface.co/j-hartmann/emotion-english-roberta-large

**Note**: Since this engineering method is only implemented to guarantee the 4 implementations necessary for the project requirements, we will work with the whole dataset (without train_test split). We will not use this method in later stages.

In [ ]:
# Using lemmatized dataset
df_train = pd.read_csv("train_clean_merged_lm.csv")

In [ ]:
notebook_login()

In [ ]:
def get_embeddings_roberta(dataframe:pd.DataFrame, model_name:str="j-hartmann/emotion-english-roberta-large", max_length:int=512):
    
    copy_df = dataframe.copy()
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    model = AutoModel.from_pretrained(model_name).to(device)

    def get_actual_embeddings(text,model_obj, tokenizer_obj, max_length=512):

        inputs = tokenizer_obj(text, return_tensors="pt", truncation=True, padding=True, max_length=max_length).to(device)
        outputs = model_obj(**inputs)
        mean_embeddings = outputs.last_hidden_state.mean(dim=1).detach().cpu().numpy()
        
        return mean_embeddings
    
    mean_embeddings_dict = {}
    
    # use tqdm to know how much time left
    for row_index, row in tqdm(copy_df.iterrows(), total=len(copy_df)):
        listing_id = row["listing_id"]
        comment = str(row["comments"])
        unlisted = row["unlisted"]
        mean_embeddings = get_actual_embeddings(comment, model, tokenizer, max_length)
        mean_embeddings_dict[listing_id] = {"embeddings":mean_embeddings, "target":unlisted}
    # save the embeddings on a column

    return mean_embeddings_dict

In [ ]:
# train set
train_embeddings_roberta = get_embeddings_roberta(df_train, model_name="j-hartmann/emotion-english-roberta-large", max_length=512)

# test set
test_embeddings_roberta = get_embeddings_roberta(df_test, model_name="j-hartmann/emotion-english-roberta-large", max_length=512)


# prepare embeddings for models
train_embeddings_roberta_models = np.array([train_embeddings_roberta[key]["embeddings"] for key in train_embeddings_roberta.keys()])
train_embeddings_roberta_models = train_embeddings_roberta_models.reshape(train_embeddings_roberta_models.shape[0], train_embeddings_roberta_models.shape[2])

# target for sklearn
target_embeddings_roberta_models = np.array([test_embeddings_roberta[key]["target"] for key in test_embeddings_roberta.keys()])
target_embeddings_roberta_models = target_embeddings_roberta_models.reshape(-1,1)

In [ ]:
train_embeddings_roberta.shape, test_embeddings_roberta.shape, train_embeddings_roberta_models.shape, target_embeddings_roberta_models.shape

##### Performing one experiment only with this transformer

In [ ]:
lr_roberta_model = LogisticRegression(class_weight="balanced", random_state=42, n_jobs=-1)

lr_roberta_model.fit(train_embeddings_roberta_models, target_embeddings_roberta_models)

# Evaluate
lr_roberta_model_predictions = lr_roberta_model.predict(test_embeddings_roberta)

In [ ]:
# Classification report
lr_robert_model_report = classification_report(test_embeddings_roberta, lr_roberta_model_predictions, target_names=["listed", "unlisted"])

print("LOGISTIC REGRESSION WITH EMBEDDINGS (MEAN BERT) REPORT\n \n")

print(lr_robert_model_report)

plot_conf_matrix(test_embeddings_roberta, lr_roberta_model_predictions, labels=["Not Unlisted", "Unlisted"], plot_name="Lemmatized Roberta Logistic Regression Confusion Matrix")

### From this part onwards, as the project requirements expect us to experiment different combinations of preprocessing, engineering, and classification methods, the notebook will branch out into 3 parts.

> ##### Part 1: Lemmatization + TF-IDF + Classification (All models)
> ##### Part 2: Lemmatization + GloVe + Classification (All models)
> ##### Part 3: Stemming + Transformer (BERT) + Classification (All models)
> ##### Part 4: Evaluation: Choose best model out of all combinations based on F1 score, and apply to test set for predictions.

# Part 1: Lemmatization + TF-IDF + Classification (All models)

# TF-IDF

In [ ]:
df_train = pd.read_csv("df_train_split_lm.csv")
df_test = pd.read_csv("df_test_split_lm.csv")

In [ ]:
tf_idf = TfidfVectorizer(max_features=15000,smooth_idf=True, stop_words="english", sublinear_tf=True)

In [ ]:
X_data_train, y_data_train = tf_idf.fit_transform(df_train["comments"]), df_train["unlisted"]

X_data_test, y_data_test = tf_idf.transform(df_test["comments"]), df_test["unlisted"]

In [ ]:
X_data_train.shape, X_data_test.shape, y_data_train.shape, y_data_test.shape

# Models

In [ ]:
# Logistic Regression

lr = LogisticRegression(class_weight="balanced", random_state=42, n_jobs=-1)

lr.fit(X_data_train, y_data_train)

predictions_lr = lr.predict(X_data_test)

lr_classif_report = classification_report(y_data_test, predictions_lr)


print("\n \n Lemmatized TF-IDF Logistic Regression Classifier \n \n")

print(lr_classif_report)

print("\n \n")

plot_conf_matrix(y_data_test, predictions_lr, labels=["Not Unlisted", "Unlisted"], plot_name="Lemmatized TF-IDF Logistic Regression Classifier")

In [ ]:
# K-Nearest Neighbors

knn = KNeighborsClassifier(n_neighbors=5, n_jobs=-1)

knn.fit(X_data_train, y_data_train)

predictions_knn = knn.predict(X_data_test)

knn_classif_report = classification_report(y_data_test, predictions_knn)


print("\n \n Lemmatized TF-IDF KNN Classifier \n \n")

print(knn_classif_report)

print("\n \n")

plot_conf_matrix(y_data_test, predictions_knn, labels=["Not Unlisted", "Unlisted"], plot_name="Lemmatized TF-IDF KNN Classifier")

In [ ]:
# Stochastic Gradient Descent

sgd = SGDClassifier(loss="hinge", penalty="l2", max_iter=1000, class_weight="balanced",\
     random_state=42, fit_intercept=False, tol=0.001, early_stopping=True, validation_fraction=0.1, \
        n_iter_no_change=5, n_jobs=-1, verbose=1, shuffle=True, learning_rate="adaptive", eta0=0.000001)

sgd.fit(X_data_train, y_data_train)

predictions_sgd = sgd.predict(X_data_test)

sgd_classif_report = classification_report(y_data_test, predictions_sgd)

In [ ]:
print("\n \n Lemmatized TF-IDF SGD Classifier \n \n")

print(sgd_classif_report)

print("\n \n")

plot_conf_matrix(y_data_test, predictions_sgd, labels=["Not Unlisted", "Unlisted"], plot_name="Lemmatized TF-IDF SGD Classifier")

In [ ]:
# Multi-Layer Perceptron (MLP)

mlp = MLPClassifier(solver='adam', hidden_layer_sizes=(2,2), activation='logistic', random_state=42)

mlp.fit(X_data_train, y_data_train)

predictions_mlp = mlp.predict(X_data_test)

mlp_classif_report = classification_report(y_data_test, predictions_mlp)


print("\n \n Lemmatized TF-IDF MLP Classifier \n \n")

print(mlp_classif_report)

print("\n \n")

plot_conf_matrix(y_data_test, predictions_mlp, labels=["Not Unlisted", "Unlisted"], plot_name="Lemmatized TF-IDF MLP Classifier")

#### Other Advanced Models - Random Forest and HistGradientBoosting

In [ ]:
# Random Forest

rf_classifier = RandomForestClassifier(random_state=42, n_jobs=-1, class_weight="balanced", verbose=1)

rf_classifier.fit(X_data_train, y_data_train)

predictions_rf = rf_classifier.predict(X_data_test)

rf_classif_report = classification_report(y_data_test, predictions_rf)


print("\n \n Lemmatized TF-IDF Random Forest Classifier \n \n")

print(rf_classif_report)

print("\n \n")

plot_conf_matrix(y_data_test, predictions_rf, labels=["Not Unlisted", "Unlisted"], plot_name="Lemmatized TF-IDF Random Forest Classifier")

In [ ]:
# HistGradientBoosting

hist_grad_boost = HistGradientBoostingClassifier(random_state=42, verbose=1)

hist_grad_boost.fit(X_data_train.toarray(), y_data_train)

predictions_histgradboost = hist_grad_boost.predict(X_data_test.toarray())

histgradboost_classif_report = classification_report(y_data_test, predictions_histgradboost)

In [ ]:
print("\n \n Lemmatized TF-IDF HistGradientBoosting Classifier \n \n")

print(histgradboost_classif_report)

print("\n \n")

plot_conf_matrix(y_data_test, predictions_histgradboost, labels=["Not Unlisted", "Unlisted"], plot_name="Lemmatized TF-IDF Random Forest Classifier")

# Part 2: Lemmatization + GloVe + Classification (All models)

# GloVe

In [ ]:
df_train = pd.read_csv("df_train_split_lm.csv")
df_test = pd.read_csv("df_test_split_lm.csv")

In [ ]:
glove = vocab.GloVe(name='840B', dim=300)
print(f'Loaded {len(glove.itos)} words')

In [ ]:
def from_sentence_to_vec_840B(sentence_to_vectorize):
    sentence_vec = []
    for word in sentence_to_vectorize:
        try:
            word_vec = glove.get_vecs_by_tokens(word, lower_case_backup=True)
            sentence_vec.append(word_vec)
        except KeyError:
            pass
    sentence_mean = np.mean(sentence_vec, axis=0)
    return sentence_mean

In [ ]:
train_comments_list = df_train["comments"].tolist()
train_comments_list = [sentence.split() for sentence in train_comments_list]

test_comments_list = df_test["comments"].tolist()
test_comments_list = [sentence.split() for sentence in test_comments_list]

In [ ]:
sentence_vecs_train = []
sentence_vecs_test = []

# train
for i in tqdm(range(len(train_comments_list))):
    sentence_vec = from_sentence_to_vec_840B(train_comments_list[i])
    sentence_vecs_train.append(sentence_vec)

# test
for i in tqdm(range(len(test_comments_list))):
    sentence_vec = from_sentence_to_vec_840B(test_comments_list[i])
    sentence_vecs_test.append(sentence_vec)

In [ ]:
sentence_vecs_train = torch.tensor(sentence_vecs_train)
sentence_vecs_test = torch.tensor(sentence_vecs_test)

target_train = torch.tensor(df_train["unlisted"].tolist())
target_test = torch.tensor(df_test["unlisted"].tolist())

# Models

In [ ]:
# Logistic Regression

lr = LogisticRegression(class_weight="balanced", random_state=42, n_jobs=-1)

lr.fit(sentence_vecs_train, target_train)

predictions_lr = lr.predict(sentence_vecs_test)

lr_classif_report = classification_report(target_test, predictions_lr)


print("\n \n Lemmatized GloVe Logistic Regression Classifier \n \n")

print(lr_classif_report)

print("\n \n")

plot_conf_matrix(target_test, predictions_lr, labels=["Not Unlisted", "Unlisted"], plot_name="Lemmatized GloVe Logistic Regression Classifier")


In [ ]:
# K-Nearest Neighbors

knn = KNeighborsClassifier(n_neighbors=5, n_jobs=-1)

knn.fit(sentence_vecs_train, target_train)

predictions_knn = knn.predict(sentence_vecs_test)

knn_classif_report = classification_report(target_test, predictions_knn)


print("\n \n Lemmatized GloVe KNN Classifier \n \n")

print(knn_classif_report)

print("\n \n")

plot_conf_matrix(target_test, predictions_knn, labels=["Not Unlisted", "Unlisted"], plot_name="Lemmatized GloVe KNN Classifier")

In [ ]:
# Stochastic Gradient Descent

sgd = SGDClassifier(loss="hinge", penalty="l2", max_iter=1000, class_weight="balanced",\
     random_state=42, fit_intercept=False, tol=0.001, early_stopping=True, validation_fraction=0.1, \
        n_iter_no_change=5, n_jobs=-1, verbose=1, shuffle=True, learning_rate="adaptive", eta0=0.000001)

sgd.fit(sentence_vecs_train, target_train)

predictions_sgd = sgd.predict(sentence_vecs_test)

sgd_classif_report = classification_report(target_test, predictions_sgd)


print("\n \n Lemmatized GloVe SGD Classifier \n \n")

print(sgd_classif_report)

print("\n \n")

plot_conf_matrix(target_test, predictions_sgd, labels=["Not Unlisted", "Unlisted"], plot_name="Lemmatized GloVe SGD Classifier")

In [ ]:
# Multi-Layer Perceptron (MLP)

mlp = MLPClassifier(solver='adam', hidden_layer_sizes=(2,2), activation='logistic', random_state=42)

mlp.fit(sentence_vecs_train, target_train)

predictions_mlp = mlp.predict(sentence_vecs_test)

mlp_classif_report = classification_report(target_test, predictions_mlp)


print("\n \n Lemmatized GloVe MLP Classifier \n \n")

print(mlp_classif_report)

print("\n \n")

plot_conf_matrix(target_test, predictions_mlp, labels=["Not Unlisted", "Unlisted"], plot_name="Lemmatized GloVe MLP Classifier")

#### Other Advanced Models - Random Forest and HistGradientBoosting

In [ ]:
# Random Forest

rf_classifier = RandomForestClassifier(random_state=42, n_jobs=-1, class_weight="balanced", verbose=1)

rf_classifier.fit(sentence_vecs_train, target_train)

predictions_rf = rf_classifier.predict(sentence_vecs_test)

rf_classif_report = classification_report(target_test, predictions_rf)



print("\n \n Lemmatized GloVe Random Forest Classifier \n \n")

print(rf_classif_report)

print("\n \n")

plot_conf_matrix(target_test, predictions_rf, labels=["Not Unlisted", "Unlisted"], plot_name="Lemmatized GloVe Random Forest Classifier")

In [ ]:
# HistGradientBoosting

hist_grad_boost = HistGradientBoostingClassifier(random_state=42, verbose=1)

hist_grad_boost.fit(sentence_vecs_train, target_train)

predictions_histgradboost = hist_grad_boost.predict(sentence_vecs_test)

histgradboost_classif_report = classification_report(target_test, predictions_histgradboost)


print("\n \n Lemmatized GloVe HistGradientBoosting Classifier \n \n")

plot_conf_matrix(target_test, predictions_histgradboost, labels=["Not Unlisted", "Unlisted"], plot_name="Lemmatized GloVe HistGradientBoosting Classifier")

In [ ]:
# Classification report
print("\n \n")
print(histgradboost_classif_report)

# Part 3 - Stemming + BERT + Classification (All models)

https://huggingface.co/google-bert/bert-base-uncased

In [ ]:
df_train = pd.read_csv("df_train_split_stm.csv")
df_test = pd.read_csv("df_test_split_stm.csv")

## BERT

In [ ]:
notebook_login()

In [ ]:
def get_embeddings_bert(dataframe:pd.DataFrame, model_name:str="google-bert/bert-base-uncased", max_length:int=512):
    copy_df = dataframe.copy()
    
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    model = AutoModel.from_pretrained(model_name).to(device)

    def get_actual_embeddings(text,model_obj, tokenizer_obj, max_length=512):

        inputs = tokenizer_obj(text, return_tensors="pt", truncation=True, padding=True, max_length=max_length).to(device)
        outputs = model_obj(**inputs)
        cls_embeddings = outputs.last_hidden_state[:,0,:].detach().cpu().numpy()
        mean_embeddings = outputs.last_hidden_state.mean(dim=1).detach().cpu().numpy()

        return cls_embeddings, mean_embeddings
    
    cls_embeddings_dict = {}
    mean_embeddings_dict = {}

    # use tqdm to know how much time left
    for row_index, row in tqdm(copy_df.iterrows(), total=len(copy_df)):
        listing_id = row["listing_id"]
        comment = str(row["comments"])
        unlisted = row["unlisted"]

        cls_embeddings, mean_embeddings = get_actual_embeddings(comment, model, tokenizer, max_length)
        cls_embeddings_dict[listing_id] = {"embeddings":cls_embeddings, "target":unlisted}
        mean_embeddings_dict[listing_id] = {"embeddings":mean_embeddings, "target":unlisted}
        
    # save the embeddings on a column

    return cls_embeddings_dict, mean_embeddings_dict

In [ ]:
cls_train_embeddings, mean_train_embeddings = get_embeddings_bert(df_train, model_name="google-bert/bert-base-uncased", max_length=512)

In [ ]:
cls_test_embeddings, mean_test_embeddings = get_embeddings_bert(df_test, model_name="google-bert/bert-base-uncased", max_length=512)

In [ ]:
# prepare embeddings for models - TRAIN
cls_train_embeddings = np.array([cls_train_embeddings[key]["embeddings"] for key in cls_train_embeddings.keys()])
cls_train_embeddings = cls_train_embeddings.reshape(cls_train_embeddings.shape[0], mean_train_embeddings.shape[2])

# target
cls_train_embeddings_target = np.array([cls_train_embeddings[key]["target"] for key in cls_train_embeddings.keys()])
cls_train_embeddings_target = cls_train_embeddings_target.reshape(-1,1)

In [ ]:
# prepare embeddings for models - TEST
cls_test_embeddings = np.array([cls_test_embeddings[key]["embeddings"] for key in cls_test_embeddings.keys()])
cls_test_embeddings = cls_test_embeddings.reshape(cls_test_embeddings.shape[0], mean_train_embeddings.shape[2])

# target
cls_test_embeddings_target = np.array([cls_test_embeddings[key]["target"] for key in cls_train_embeddings.keys()])
cls_test_embeddings_target = cls_test_embeddings_target.reshape(-1,1)

In [ ]:
cls_train_embeddings.shape, cls_train_embeddings_target.shape

In [ ]:
cls_test_embeddings.shape, cls_test_embeddings_target.shape

# Models

In [ ]:
# Logistic Regression

lr = LogisticRegression(class_weight="balanced", random_state=42, n_jobs=-1)

lr.fit(cls_train_embeddings, cls_train_embeddings_target)

predictions_lr = lr.predict(cls_test_embeddings)

lr_classif_report = classification_report(cls_test_embeddings_target, predictions_lr)


print("\n \n Stemmed BERT Logistic Regression Classifier \n \n")

print(lr_classif_report)

print("\n \n")

plot_conf_matrix(cls_test_embeddings_target, predictions_lr, labels=["Not Unlisted", "Unlisted"], plot_name="Stemmed BERT Logistic Regression Classifier")


In [ ]:
# K-Nearest Neighbors

knn = KNeighborsClassifier(n_neighbors=5, n_jobs=-1)

knn.fit(cls_train_embeddings, cls_train_embeddings_target)

predictions_knn = knn.predict(cls_test_embeddings)

knn_classif_report = classification_report(cls_test_embeddings_target, predictions_knn)


print("\n \n Stemmed BERT KNN Classifier \n \n")

print(knn_classif_report)

print("\n \n")

plot_conf_matrix(cls_test_embeddings_target, predictions_knn, labels=["Not Unlisted", "Unlisted"], plot_name="Stemmed BERT KNN Classifier")

In [ ]:
# Stochastic Gradient Descent

sgd = SGDClassifier(loss="hinge", penalty="l2", max_iter=1000, class_weight="balanced",\
     random_state=42, fit_intercept=False, tol=0.001, early_stopping=True, validation_fraction=0.1, \
        n_iter_no_change=5, n_jobs=-1, verbose=1, shuffle=True, learning_rate="adaptive", eta0=0.000001)

sgd.fit(cls_train_embeddings, cls_train_embeddings_target)

predictions_sgd = sgd.predict(cls_test_embeddings)

sgd_classif_report = classification_report(cls_test_embeddings_target, predictions_sgd)


print("\n \n Stemmed BERT SGD Classifier \n \n")

print(sgd_classif_report)

print("\n \n")

plot_conf_matrix(cls_test_embeddings_target, predictions_sgd, labels=["Not Unlisted", "Unlisted"], plot_name="Stemmed BERT SGD Classifier")

In [ ]:
# Multi-Layer Perceptron (MLP)

mlp = MLPClassifier(solver='adam', hidden_layer_sizes=(2,2), activation='logistic', random_state=42)

mlp.fit(cls_train_embeddings, cls_train_embeddings_target)

predictions_mlp = mlp.predict(cls_test_embeddings)

mlp_classif_report = classification_report(cls_test_embeddings_target, predictions_mlp)


print("\n \n Stemmed BERT SGD Classifier \n \n")

print(mlp_classif_report)

print("\n \n")

plot_conf_matrix(cls_test_embeddings_target, predictions_mlp, labels=["Not Unlisted", "Unlisted"], plot_name="Stemmed BERT MLP Classifier")

#### Other Advanced Models - Random Forest and HistGradientBoosting

In [ ]:
# Random Forest

rf_classifier = RandomForestClassifier(random_state=42, n_jobs=-1, class_weight="balanced", verbose=1)

rf_classifier.fit(cls_train_embeddings, cls_train_embeddings_target)

predictions_rf = rf_classifier.predict(cls_test_embeddings)

rf_classif_report = classification_report(cls_test_embeddings_target, predictions_rf)


print("\n \n Stemmed BERT Random Forest Classifier \n \n")

print(rf_classif_report)

print("\n \n")

plot_conf_matrix(cls_test_embeddings_target, predictions_rf, labels=["Not Unlisted", "Unlisted"], plot_name="Stemmed BERT Random Forest Classifier")

In [ ]:
# HistGradientBoosting

hist_grad_boost = HistGradientBoostingClassifier(random_state=42, verbose=1)

hist_grad_boost.fit(cls_train_embeddings.toarray(), cls_train_embeddings_target)

predictions_histgradboost = hist_grad_boost.predict(cls_test_embeddings.toarray())

histgradboost_classif_report = classification_report(cls_test_embeddings_target, predictions_histgradboost)


print("\n \n Stemmed BERT HistGradientBoosting Classifier \n \n")

print(histgradboost_classif_report)

print("\n \n")

plot_conf_matrix(cls_test_embeddings_target, predictions_histgradboost, labels=["Not Unlisted", "Unlisted"], plot_name="Stemmed BERTF Random Forest Classifier")

# Part 4: Prediction on test set with best model:  Lemmatized TF-IDF MLP Classifier

In [ ]:
df_test = pd.read_csv("test_clean_merged.csv")

In [ ]:
# Lemmatizing
updates = lemmatization(df_test["comments"])

update_df(df_test, updates)

In [ ]:
# Applying TF-IDF
test_data = tf_idf.transform(df_test["comments"])

In [ ]:
# Generating predictions
predictions_mlp = mlp.predict(test_data)

In [ ]:
predictions_mlp

In [ ]:
df_test.shape

In [ ]:
len(predictions_mlp)

In [ ]:
df_test.columns

In [ ]:
df_test["predicted"] = predictions_mlp

In [ ]:
df_test.head()

In [ ]:
columns = ["listing_id","predicted"]

In [ ]:
df_test = df_test[columns]

In [ ]:
df_test

In [ ]:
df_test.rename(columns={"listing_id":"id"}, inplace=True)

In [ ]:
df_test.to_csv("Predictions_15.csv", sep = ",", index = False)

In [ ]:
test = pd.read_csv("Predictions_15.csv")

In [ ]:
test